# Data Cleaning

In [24]:
import pandas as pd
import numpy as np

## Load Dataset

In [25]:
original_data = pd.read_csv('data/raw.csv', low_memory=False)
data = original_data.copy()

## Remove Duplicate Rows

In [26]:
duplicates = data[data.duplicated(keep=False)]
duplicates.sort_values("id").head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
676,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
1465,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
14012,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",http://www.dealthemovie.com/,11115,tt0446676,en,Deal,As an ex-gambler teaches a hot-shot college ki...,...,2008-01-29,0.0,85.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Deal,False,5.2,22.0
24844,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",http://www.dealthemovie.com/,11115,tt0446676,en,Deal,As an ex-gambler teaches a hot-shot college ki...,...,2008-01-29,0.0,85.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Deal,False,5.2,22.0
19890,False,NaN,0,"[{'id': 14, 'name': 'Fantasy'}, {'id': 18, 'na...",NaN,119916,tt0080000,en,The Tempest,"Prospero, the true Duke of Milan is now living...",...,1980-02-27,0.0,123.0,[],Released,NaN,The Tempest,False,0.0,0.0


In [27]:
data = data.drop_duplicates().reset_index(drop=True)
print("duplicate data delete!")
print(f"Duplicate rows after cleaning: {data.duplicated().sum()}")

duplicate data delete!
Duplicate rows after cleaning: 0


## Fixed Data Type

### Data Type Issues (9 Columns Need Conversion)

| Column | Current → Expected |
|--------|-------------------|
| `adult`, `video` | object → bool |
| `budget`, `id`, `popularity` | object → int64/float64 |
| `revenue`, `runtime`, `vote_count` | float64 → int64 |
| `release_date` | object → datetime64[ns] |

In [28]:
numeric_columns = [
    "budget",
    "id",
    "popularity",
    "revenue",
    "runtime",
    "vote_average",
    "vote_count"
]
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    print(f"type of {col} changed to {data[col].dtype}")

type of budget changed to float64
type of id changed to float64
type of popularity changed to float64
type of revenue changed to float64
type of runtime changed to float64
type of vote_average changed to float64
type of vote_count changed to float64


In [29]:
data["release_date"] = pd.to_datetime(data["release_date"], errors="coerce")
print("type of {} changed to {}".format("release_date", data["release_date"].dtype))

type of release_date changed to datetime64[ns]


### Clean Boolean Columns

3 rows in `adult` column contain text/overview data (likely import shift)

In [30]:
invalid_data = data[~data["adult"].isin(['True', 'False'])]
invalid_data

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19725,- Written by Ørnås,0.065736,NaN,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",NaN,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29491,Rune Balot goes to a casino connected to the ...,1.931659,NaN,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",NaN,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35575,Avalanche Sharks tells the story of a bikini ...,2.185485,NaN,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",NaN,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Remove corrupted rows caused by CSV parsing issues
data = data[data["adult"].isin(['True', 'False'])]
print(f"Invalid rows in 'adult' column: {len(data[~data['adult'].isin(['True', 'False'])])}")

Invalid rows in 'adult' column: 0


In [32]:
boolean_col = ["video", "adult"]
for col in boolean_col:
    data[col] = data[col].map({'True': True, 'False': False}).astype('boolean')
    print(f"type of {col} changed to {data[col].dtype}")

type of video changed to boolean
type of adult changed to boolean


### Cleaning Summary

- Removed 3 corrupted rows caused by CSV parsing issues.
- Converted `adult` and `video` to Boolean dtype.
- Dataset is now consistent for Boolean features.

## Parse JSON Columns

In [33]:
# Check the structure of a sample row
sample = data["genres"].iloc[0]
print(sample)
print(type(sample))

[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]
<class 'str'>


In [34]:
import ast

# Convert string to list of dictionaries
data['genres'] = data['genres'].apply(ast.literal_eval)
sample = data["genres"].iloc[0]